# Notebook 04: XGBoost Fatigue & Injury Risk Model
## FixtureIQ — Multi-Season, Multi-Team Analysis (2022/23 – 2024/25)

### Objectives
1. **Upgrade from Logistic Regression → XGBoost** for better non-linear pattern capture
2. **Multi-season data** (22/23, 23/24, 24/25) to solve the single-season sample-size problem flagged in Notebook 03
3. **Control group comparison**: UCL teams (Arsenal, Liverpool, Man City, Aston Villa) vs. non-UCL PL teams (Brighton, Brentford, Everton, Wolves) — direct test of congestion effect
4. **Opponent quality (ELO)** as an extra predictor, addressing the tutor's missing-variables note
5. **Hierarchical structure**: team-level random effects via grouped cross-validation, aligned with tutor's Bayesian hierarchical suggestion

### No-Leakage Rule (same as Notebook 03)
All predictors must be **pre-match only** — no same-match outcomes used as features.

### Target Variable
`decline_flag` = 1 if player's `overall_score` in match M is ≥10% below their rolling 5-match baseline, else 0.

## 0. Install / import dependencies

In [1]:
# Install if needed (uncomment first time)
# !pip install xgboost shap scikit-learn pandas numpy matplotlib seaborn soccerdata --quiet

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Modelling
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve, ConfusionMatrixDisplay
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils.class_weight import compute_sample_weight

# Explainability
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('SHAP not installed — install with: pip install shap')

print('Libraries loaded successfully.')
print(f'XGBoost version: {xgb.__version__}')

# ── Viz settings ──────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

TEAM_COLORS = {
    'Arsenal':          '#EF0107',
    'Liverpool':        '#C8102E',
    'Manchester City':  '#6CABDD',
    'Aston Villa':      '#670E36',
    'Brighton':         '#0057B8',
    'Brentford':        '#FFD700',
    'Everton':          '#003399',
    'Wolves':           '#FDB913',
}

Libraries loaded successfully.
XGBoost version: 3.2.0


## 1. Configuration — paths, seasons, teams

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Adjust DATA_BASE to wherever your FBref pipeline saved its output
DATA_BASE = Path('/home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data')
OUTPUT_DIR = DATA_BASE / 'XGBOOST_MODEL'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = OUTPUT_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

# ── Seasons ───────────────────────────────────────────────────────────────────
SEASONS = ['2022_2023', '2023_2024', '2024_2025']
SEASON_DIRS = {s: DATA_BASE / f'SEASON_{s}' for s in SEASONS}

# ── UCL cohort (congested) ────────────────────────────────────────────────────
UCL_TEAMS = {
    'Arsenal':          'arsenal',
    'Liverpool':        'liverpool',
    'Manchester City':  'manchester_city',
    'Aston Villa':      'aston_villa',   # Only in UCL for 24/25
}

# ── Control group (non-UCL PL teams) ─────────────────────────────────────────
# These teams only play PL + domestic cups → lower fixture load → control group
# Choose teams with stable squad size and no regular European football
CONTROL_TEAMS = {
    'Brighton':   'brighton',
    'Brentford':  'brentford',
    'Everton':    'everton',
    'Wolves':     'wolves',
}

ALL_TEAMS = {**UCL_TEAMS, **CONTROL_TEAMS}

# ── FBref squad IDs ───────────────────────────────────────────────────────────
# Used to build football_data_pipeline URLs for control teams if not yet downloaded
SQUAD_IDS = {
    'Arsenal':         '18bb7c10',
    'Liverpool':       '822bd0ba',
    'Manchester City': 'b8fd03ef',
    'Aston Villa':     '8602292d',
    'Brighton':        'd07537b9',
    'Brentford':       'cd051869',
    'Everton':         'd3fd31cc',
    'Wolves':          '8cec06e1',
}

# ── Model params ─────────────────────────────────────────────────────────────
DECLINE_THRESHOLD = 0.10    # 10% below rolling baseline = decline
MIN_MINUTES       = 30      # Minimum minutes for a match to be included
ROLLING_WINDOW    = 5       # Matches used for rolling baseline
RANDOM_STATE      = 42

print('Configuration set.')
print(f'  UCL teams   : {list(UCL_TEAMS.keys())}')
print(f'  Control teams: {list(CONTROL_TEAMS.keys())}')
print(f'  Seasons     : {SEASONS}')

Configuration set.
  UCL teams   : ['Arsenal', 'Liverpool', 'Manchester City', 'Aston Villa']
  Control teams: ['Brighton', 'Brentford', 'Everton', 'Wolves']
  Seasons     : ['2022_2023', '2023_2024', '2024_2025']


## 2. Load & unify match-report player data across all seasons and teams

This cell reads from the `match_reports/*/player_stats.csv` files produced by `football_data_pipeline.py`.
It expects columns: `Player`, `Min`, `Gls`, `Ast`, `Sh`, `SoT`, `Crs`, `Int`, `TklWon`, etc.
Adapt column names below if your pipeline outputs slightly different headers.

In [ ]:
def load_match_reports(team_slug: str, season: str, season_dir: Path, team_name: str) -> pd.DataFrame:
    """Load all per-match player stat CSVs for one team × season."""
    team_dir = season_dir / f'{team_slug}_{season}'
    reports_glob = list(team_dir.glob('match_reports/*/player_stats.csv'))
    
    if not reports_glob:
        # Try alternative naming from pipeline
        reports_glob = list(team_dir.glob('match_reports/*/*_player_stats.csv'))
    
    if not reports_glob:
        print(f'  WARNING: No match reports found for {team_name} {season}')
        return pd.DataFrame()
    
    dfs = []
    for path in sorted(reports_glob):
        # Extract date and opponent from path
        parts = path.parent.name.split('_')   # e.g. 2024-08-17_premier_league_wolves
        match_date = parts[0] if parts else 'unknown'
        competition = parts[1] + '_' + parts[2] if len(parts) > 2 else 'unknown'
        opponent    = '_'.join(parts[3:]) if len(parts) > 3 else 'unknown'
        
        df = pd.read_csv(path)
        df['match_date']  = match_date
        df['competition'] = competition
        df['opponent']    = opponent
        df['match_id']    = path.parent.name
        dfs.append(df)
    
    if not dfs:
        return pd.DataFrame()
    
    out = pd.concat(dfs, ignore_index=True)
    out['team']   = team_name
    out['season'] = season
    out['is_ucl_team'] = int(team_name in UCL_TEAMS)
    return out


# ── Load all data ─────────────────────────────────────────────────────────────
raw_frames = []

for season in SEASONS:
    season_dir = SEASON_DIRS[season]
    if not season_dir.exists():
        print(f'  MISSING season dir: {season_dir}')
        continue
    for team_name, team_slug in ALL_TEAMS.items():
        df = load_match_reports(team_slug, season, season_dir, team_name)
        if not df.empty:
            raw_frames.append(df)
            print(f'  Loaded {team_name} {season}: {len(df)} player-match rows')

if not raw_frames:
    raise RuntimeError('No data loaded — check DATA_BASE path and folder structure.')

raw = pd.concat(raw_frames, ignore_index=True)
print(f'\nTotal raw rows : {len(raw):,}')
print(f'Columns        : {raw.columns.tolist()}')

## 3. Data cleaning & column standardisation

In [ ]:
df = raw.copy()

# ── Column normalisation ──────────────────────────────────────────────────────
# Map common FBref column variants to standardised names
RENAME_MAP = {
    'Player':   'player', 'player_name': 'player',
    'Min':      'minutes', 'Mins': 'minutes',
    'Pos':      'position', 'Position': 'position',
    'Gls':      'goals', 'Goals': 'goals',
    'Ast':      'assists', 'Assists': 'assists',
    'Sh':       'shots',   'Shots': 'shots',
    'SoT':      'shots_on_target',
    'SoT%':     'sot_pct',
    'xG':       'xg',
    'xAG':      'xag',
    'ProgC':    'prog_carries', 'ProgP': 'prog_passes',
    'Crs':      'crosses',
    'Int':      'interceptions',
    'TklWon':   'tackles_won',
    'Tkl':      'tackles',
    'Press':    'pressures',
    'Clr':      'clearances',
    'Err':      'errors',
    'YC':       'yellow_cards', 'CrdY': 'yellow_cards',
    'RC':       'red_cards',    'CrdR': 'red_cards',
}
df = df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})

# ── Types ──────────────────────────────────────────────────────────────────────
df['match_date'] = pd.to_datetime(df['match_date'], errors='coerce')
df['minutes']    = pd.to_numeric(df.get('minutes', 0), errors='coerce').fillna(0)

NUMERIC_COLS = [
    'goals','assists','shots','shots_on_target','xg','xag',
    'prog_carries','prog_passes','crosses','interceptions',
    'tackles_won','tackles','pressures','clearances',
    'yellow_cards','red_cards','errors',
]
for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# ── Filter: meaningful playing time only ──────────────────────────────────────
df = df[df['minutes'] >= MIN_MINUTES].copy()

# ── Position groups ────────────────────────────────────────────────────────────
def map_position(pos: str) -> str:
    pos = str(pos).upper()
    if 'GK' in pos:                          return 'GK'
    if any(x in pos for x in ['CB','RB','LB','WB','DF']): return 'DF'
    if 'DM' in pos:                          return 'DM'
    if any(x in pos for x in ['AM','LW','RW']): return 'AM'
    if any(x in pos for x in ['FW','ST','CF']): return 'FW'
    return 'MF'

if 'position' in df.columns:
    df['position_group'] = df['position'].apply(map_position)
else:
    df['position_group'] = 'MF'

# ── Competition type ─────────────────────────────────────────────────────────
def map_competition(comp: str) -> str:
    comp = str(comp).lower()
    if any(x in comp for x in ['champions','ucl','cl']):  return 'UCL'
    if any(x in comp for x in ['premier','epl','pl']):    return 'PL'
    if 'fa_cup' in comp or 'facup' in comp:               return 'FAC'
    if any(x in comp for x in ['efl','carabao','league_cup']): return 'EFL'
    return 'OTHER'

df['competition_type'] = df['competition'].apply(map_competition)

# ── Sort chronologically per player per team ──────────────────────────────────
df = df.sort_values(['team','player','match_date']).reset_index(drop=True)

print(f'Clean dataset: {len(df):,} rows × {len(df.columns)} columns')
print(f'Players : {df["player"].nunique()}')
print(f'Seasons : {df["season"].unique().tolist()}')
print(f'Teams   : {df["team"].unique().tolist()}')

## 4. Build composite performance score

Same scoring logic as Notebook 02 — weighted by position to capture what "good performance" means for each role.

In [ ]:
# Position-weighted scoring matrix
SCORE_WEIGHTS = {
    #              goals  assists  shots  xg    xag   prog_c prog_p inter  tkl  press  clr
    'GK': dict(goals=0,  assists=0.2, shots=0,   xg=0,   xag=0.2, prog_carries=0.1, prog_passes=0.3, interceptions=0,   tackles=0.1, pressures=0,   clearances=0.1),
    'DF': dict(goals=0.3,assists=0.3, shots=0.1, xg=0.2, xag=0.2, prog_carries=0.3, prog_passes=0.5, interceptions=1.5, tackles=1.2, pressures=0.5, clearances=0.8),
    'DM': dict(goals=0.4,assists=0.5, shots=0.2, xg=0.3, xag=0.4, prog_carries=0.5, prog_passes=0.8, interceptions=1.2, tackles=1.0, pressures=0.8, clearances=0.3),
    'MF': dict(goals=0.8,assists=0.8, shots=0.4, xg=0.6, xag=0.6, prog_carries=0.7, prog_passes=0.9, interceptions=0.6, tackles=0.6, pressures=0.6, clearances=0.1),
    'AM': dict(goals=1.0,assists=1.0, shots=0.6, xg=0.8, xag=0.8, prog_carries=1.0, prog_passes=0.7, interceptions=0.3, tackles=0.3, pressures=0.4, clearances=0.0),
    'FW': dict(goals=1.5,assists=0.8, shots=0.8, xg=1.2, xag=0.6, prog_carries=0.8, prog_passes=0.4, interceptions=0.1, tackles=0.1, pressures=0.3, clearances=0.0),
}
DEFAULT_WEIGHTS = SCORE_WEIGHTS['MF']

def compute_performance_score(row: pd.Series) -> float:
    w = SCORE_WEIGHTS.get(row['position_group'], DEFAULT_WEIGHTS)
    score = 0.0
    for col, weight in w.items():
        score += row.get(col, 0) * weight
    # Normalise by minutes so 90-min appearances are comparable
    mins = max(row.get('minutes', 1), 1)
    return (score / mins) * 90.0

df['perf_score'] = df.apply(compute_performance_score, axis=1)

# ── Rolling baseline (5-match per player) ────────────────────────────────────
df['rolling_baseline'] = (
    df.groupby(['team','player'])['perf_score']
      .transform(lambda x: x.shift(1).rolling(ROLLING_WINDOW, min_periods=2).mean())
)

# ── Decline flag (target) ─────────────────────────────────────────────────────
df['decline_flag'] = (
    (df['perf_score'] < df['rolling_baseline'] * (1 - DECLINE_THRESHOLD)) &
    df['rolling_baseline'].notna()
).astype(int)

# Drop rows with no baseline (first few matches of each player)
df = df[df['rolling_baseline'].notna()].copy()

decline_rate = df['decline_flag'].mean()
print(f'Decline flag rate: {decline_rate:.1%}')
print(f'Dataset after filtering: {len(df):,} rows')

# Quick sanity check per group
print('\nDecline rate by group:')
print(df.groupby('is_ucl_team')['decline_flag'].agg(['mean','count']).round(3))

## 5. Feature engineering — fatigue + context variables

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """All features must be computable before match M using only M-1, M-2, ... data."""
    df = df.sort_values(['team','player','match_date']).copy()

    # ── Rest days ─────────────────────────────────────────────────────────────
    df['rest_days'] = (
        df.groupby(['team','player'])['match_date']
          .transform(lambda x: x.diff().dt.days)
    )
    df['short_rest']   = (df['rest_days'] <= 4).astype(int)
    df['very_short_rest'] = (df['rest_days'] <= 3).astype(int)

    # ── Workload rolling windows ───────────────────────────────────────────────
    for window in [1, 3, 5]:
        df[f'min_last_{window}'] = (
            df.groupby(['team','player'])['minutes']
              .transform(lambda x: x.shift(1).rolling(window, min_periods=1).sum())
        )
        df[f'starts_last_{window}'] = (
            df.groupby(['team','player'])['minutes']
              .transform(lambda x: (x.shift(1) >= 60).rolling(window, min_periods=1).sum())
        )

    # ── Short-rest frequency ──────────────────────────────────────────────────
    for window in [3, 5]:
        df[f'short_rest_count_{window}'] = (
            df.groupby(['team','player'])['short_rest']
              .transform(lambda x: x.shift(1).rolling(window, min_periods=1).sum())
        )

    # ── UCL-specific load ─────────────────────────────────────────────────────
    df['is_ucl_match'] = (df['competition_type'] == 'UCL').astype(int)
    df['ucl_min_last_3'] = (
        df.groupby(['team','player'])
          .apply(lambda g: (
              g[g['competition_type'] == 'UCL']['minutes']
               .reindex(g.index).shift(1).rolling(3, min_periods=0).sum()
          ))
          .reset_index(level=0, drop=True)
          .fillna(0)
    )
    df['played_ucl_last_match'] = (
        df.groupby(['team','player'])['is_ucl_match']
          .transform(lambda x: x.shift(1))
          .fillna(0).astype(int)
    )

    # ── Season load accumulation ──────────────────────────────────────────────
    df['cumulative_minutes_season'] = (
        df.groupby(['team','player','season'])['minutes']
          .transform(lambda x: x.shift(1).expanding().sum())
          .fillna(0)
    )
    df['cumulative_matches_season'] = (
        df.groupby(['team','player','season'])['minutes']
          .transform(lambda x: (x.shift(1) > 0).expanding().sum())
          .fillna(0)
    )

    # ── Previous performance ─────────────────────────────────────────────────
    df['prev_perf_score'] = (
        df.groupby(['team','player'])['perf_score']
          .transform(lambda x: x.shift(1))
    )
    df['prev_decline'] = (
        df.groupby(['team','player'])['decline_flag']
          .transform(lambda x: x.shift(1))
          .fillna(0).astype(int)
    )

    # ── Venue ─────────────────────────────────────────────────────────────────
    venue_col = next((c for c in df.columns if c.lower() in ['venue','home_away']), None)
    if venue_col:
        df['is_away'] = (df[venue_col].str.lower() == 'away').astype(int)
    else:
        df['is_away'] = 0

    # ── Season phase (early/mid/late) ──────────────────────────────────────────
    month = df['match_date'].dt.month
    df['season_phase'] = pd.cut(
        month,
        bins=[0, 10, 1, 4, 12],   # Aug-Oct / Nov-Jan / Feb-Apr / May+
        labels=['early','winter','knockouts','end'],
        ordered=False
    ).astype(str)

    # ── Congestion group label ─────────────────────────────────────────────────
    df['congestion_group'] = df['is_ucl_team'].map({1: 'UCL_team', 0: 'Control'})

    return df


df = engineer_features(df)

# Preview engineered features
feat_cols_preview = [
    'player','team','match_date','season','competition_type',
    'rest_days','short_rest','min_last_3','min_last_5',
    'ucl_min_last_3','played_ucl_last_match',
    'cumulative_minutes_season','prev_perf_score',
    'rolling_baseline','is_ucl_team','decline_flag'
]
print(df[[c for c in feat_cols_preview if c in df.columns]].head(8).to_string())

## 6. Add opponent quality — ClubElo ratings

Addresses tutor feedback: *'Opponent strength not included'*.

In [ ]:
ELO_CSV = DATA_BASE / 'fixtureiq_elo_master.csv'   # from extract_context.py

if ELO_CSV.exists():
    df_elo = pd.read_csv(ELO_CSV)
    
    # Normalise columns — soccerdata ClubElo returns (team, date, elo, rank, ...)
    df_elo.columns = df_elo.columns.str.lower().str.strip()
    if 'date' in df_elo.columns:
        df_elo['date'] = pd.to_datetime(df_elo['date'], errors='coerce')
    
    elo_col = next((c for c in df_elo.columns if 'elo' in c.lower() and 'rank' not in c.lower()), None)
    team_col = next((c for c in df_elo.columns if 'team' in c.lower() or 'club' in c.lower()), None)
    date_col = next((c for c in df_elo.columns if 'date' in c.lower()), None)
    
    if elo_col and team_col and date_col:
        df_elo = df_elo[[team_col, date_col, elo_col]].rename(
            columns={team_col: 'opp_team', date_col: 'match_date', elo_col: 'opp_elo'}
        )
        df_elo['match_date'] = pd.to_datetime(df_elo['match_date'])
        
        # Merge on opponent name + closest date
        df = pd.merge_asof(
            df.sort_values('match_date'),
            df_elo.sort_values('match_date'),
            left_on='match_date',
            right_on='match_date',
            left_by='opponent',
            right_by='opp_team',
            direction='nearest',
            tolerance=pd.Timedelta('30D')
        )
        
        pct_filled = df['opp_elo'].notna().mean()
        df['opp_elo'] = df['opp_elo'].fillna(df['opp_elo'].median())
        print(f'ELO merged — {pct_filled:.1%} of rows matched by opponent name')
    else:
        df['opp_elo'] = 1500.0   # fallback neutral ELO
        print('ELO columns not found in CSV — using neutral value 1500')
else:
    df['opp_elo'] = 1500.0
    print(f'ELO file not found at {ELO_CSV} — using neutral value 1500')
    print('  Run extract_context.py to generate it')

## 7. XGBoost model — train / evaluate

In [ ]:
# ── Feature list ─────────────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    # Rest / fatigue
    'rest_days', 'short_rest', 'very_short_rest',
    'short_rest_count_3', 'short_rest_count_5',
    # Workload
    'min_last_1', 'min_last_3', 'min_last_5',
    'starts_last_3', 'starts_last_5',
    'cumulative_minutes_season', 'cumulative_matches_season',
    # UCL specific
    'ucl_min_last_3', 'played_ucl_last_match', 'is_ucl_match',
    # Performance history
    'prev_perf_score', 'prev_decline', 'rolling_baseline',
    # Context
    'is_away', 'opp_elo', 'is_ucl_team',
]

CATEGORICAL_FEATURES = ['position_group', 'competition_type', 'season_phase']

TARGET = 'decline_flag'

# ── Encode categoricals ───────────────────────────────────────────────────────
df_model = df.dropna(subset=['rolling_baseline', 'prev_perf_score']).copy()

label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df_model[col + '_enc'] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

FEATURE_COLS = [c for c in NUMERIC_FEATURES if c in df_model.columns] + \
               [c + '_enc' for c in CATEGORICAL_FEATURES]

X = df_model[FEATURE_COLS].fillna(0)
y = df_model[TARGET]
groups = df_model['team']  # for grouped cross-validation

# ── Train / test split — time-aware ──────────────────────────────────────────
# Train on 22/23 + 23/24, test on 24/25 (respects temporal order)
train_mask = df_model['season'].isin(['2022_2023', '2023_2024'])
test_mask  = df_model['season'] == '2024_2025'

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]
groups_train = groups[train_mask]

print(f'Training set : {len(X_train):,} rows  |  Decline rate: {y_train.mean():.1%}')
print(f'Test set     : {len(X_test):,} rows   |  Decline rate: {y_test.mean():.1%}')

# ── Class imbalance — use scale_pos_weight ────────────────────────────────────
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

# ── XGBoost model ─────────────────────────────────────────────────────────────
# Hyperparameters tuned for small-medium tabular datasets with class imbalance
xgb_model = xgb.XGBClassifier(
    n_estimators        = 400,
    max_depth           = 4,          # shallow — avoids overfitting on small N
    learning_rate       = 0.05,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    scale_pos_weight    = neg_pos_ratio,   # handle class imbalance
    min_child_weight    = 5,          # regularisation
    reg_alpha           = 0.1,        # L1
    reg_lambda          = 1.5,        # L2
    eval_metric         = 'aucpr',    # optimise for precision-recall AUC
    early_stopping_rounds = 30,
    random_state        = RANDOM_STATE,
    verbosity           = 0,
)

# ── Grouped cross-validation (teams as groups) ─────────────────────────────────
# Prevents the same team from leaking between train and validation folds
# — aligns with tutor's hierarchical suggestion
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        scale_pos_weight=neg_pos_ratio, random_state=RANDOM_STATE, verbosity=0
    ),
    X_train, y_train,
    cv=sgkf,
    groups=groups_train,
    scoring='roc_auc'
)
print(f'\nGrouped 5-fold CV ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# ── Final fit with early stopping ─────────────────────────────────────────────
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred       = (y_pred_proba >= 0.35).astype(int)   # 0.35 threshold as in Notebook 03

test_auc   = roc_auc_score(y_test, y_pred_proba)
test_ap    = average_precision_score(y_test, y_pred_proba)

print(f'\nTest ROC-AUC : {test_auc:.3f}')
print(f'Test Avg Prec: {test_ap:.3f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["No decline","Decline"])}')

## 8. Evaluation plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'XGBoost (AUC={test_auc:.3f})')
axes[0].plot([0,1],[0,1],'--', color='gray', alpha=0.5, label='Random')
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve')
axes[0].legend()

# ── Precision-Recall Curve ────────────────────────────────────────────────────
prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(rec, prec, color='darkorange', lw=2, label=f'AP={test_ap:.3f}')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', alpha=0.5, label='Baseline')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')
axes[1].legend()

# ── Confusion Matrix ─────────────────────────────────────────────────────────
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['No decline', 'Decline']
).plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('Confusion Matrix (threshold=0.35)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgb_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Calibration curve ────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7,5))
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)
ax2.plot(prob_pred, prob_true, 's-', color='steelblue', label='XGBoost')
ax2.plot([0,1],[0,1], '--', color='gray', label='Perfect calibration')
ax2.set(xlabel='Mean predicted probability', ylabel='Fraction of positives',
        title='Calibration curve — are probabilities trustworthy?')
ax2.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgb_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print('A well-calibrated model hugs the diagonal — if it bows up/down, consider Platt scaling.')

## 9. Feature importance & SHAP explainability

In [ ]:
# ── Native XGBoost importance ─────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'feature':   FEATURE_COLS,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1],
               color='steelblue', alpha=0.85)
ax.set(xlabel='Feature importance (gain)', title='Top 20 XGBoost Features')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# ── SHAP values ────────────────────────────────────────────────────────────────
if HAS_SHAP:
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test)

    plt.figure(figsize=(12, 6))
    shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS,
                      plot_type='bar', show=False)
    plt.title('SHAP Summary — mean absolute impact on decline probability')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(12, 7))
    shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=False)
    plt.title('SHAP Beeswarm — direction and magnitude of each feature')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Install SHAP for explainability: pip install shap')

## 10. UCL teams vs. Control group — congestion effect analysis

This is the core research question: does Champions League fixture congestion increase decline risk, and by how much?

In [ ]:
# ── Attach predictions to full test set ──────────────────────────────────────
test_df = df_model[test_mask].copy()
test_df['decline_probability'] = y_pred_proba
test_df['predicted_decline']   = y_pred

# ── Group comparison: UCL teams vs. Control ───────────────────────────────────
group_stats = test_df.groupby('congestion_group').agg(
    n_player_matches     = ('decline_flag', 'count'),
    actual_decline_rate  = ('decline_flag', 'mean'),
    predicted_decline_rate = ('predicted_decline', 'mean'),
    avg_risk_score       = ('decline_probability', 'mean'),
    avg_rest_days        = ('rest_days', 'mean'),
    short_rest_rate      = ('short_rest', 'mean'),
    avg_min_last_3       = ('min_last_3', 'mean'),
    ucl_matches_count    = ('is_ucl_match', 'sum'),
).round(3)

print('\n=== UCL TEAMS vs. CONTROL GROUP (2024/25 Test Set) ===')
print(group_stats.T.to_string())

# ── Per-team breakdown ────────────────────────────────────────────────────────
team_stats = test_df.groupby('team').agg(
    matches         = ('decline_flag', 'count'),
    actual_declines = ('decline_flag', 'sum'),
    decline_rate    = ('decline_flag', 'mean'),
    avg_risk        = ('decline_probability', 'mean'),
    avg_rest        = ('rest_days', 'mean'),
    short_rest_pct  = ('short_rest', 'mean'),
    is_ucl          = ('is_ucl_team', 'first'),
).round(3).sort_values('avg_risk', ascending=False)

print('\n=== PER-TEAM RISK SUMMARY ===')
print(team_stats.to_string())

# ── Visualisation: decline rate by team, coloured by group ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Decline rate bar chart
colors = [TEAM_COLORS.get(t, '#888888') for t in team_stats.index]
team_stats['decline_rate'].plot.bar(ax=axes[0], color=colors, alpha=0.85, edgecolor='white')
axes[0].set(title='Actual Decline Rate by Team (2024/25)', ylabel='Decline rate', xlabel='')
axes[0].tick_params(axis='x', rotation=30)
ucl_patch    = mpatches.Patch(color='#EF0107', alpha=0.7, label='UCL teams')
ctrl_patch   = mpatches.Patch(color='#0057B8', alpha=0.7, label='Control (no UCL)')
axes[0].legend(handles=[ucl_patch, ctrl_patch])

# Rest days violin plot
sns.violinplot(
    data=test_df, x='congestion_group', y='rest_days',
    palette={'UCL_team': '#EF0107', 'Control': '#0057B8'},
    inner='quartile', ax=axes[1]
)
axes[1].set(title='Rest Days Distribution: UCL teams vs Control', ylabel='Rest days', xlabel='')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'group_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Congestion effect — risk over season timeline

In [ ]:
# ── Monthly average risk by group ─────────────────────────────────────────────
test_df['month'] = test_df['match_date'].dt.to_period('M')
monthly = (
    test_df
    .groupby(['month', 'congestion_group'])['decline_probability']
    .mean()
    .unstack('congestion_group')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(16, 5))
if 'UCL_team' in monthly.columns:
    ax.plot(monthly.index.astype(str), monthly['UCL_team'],
            'o-', color='#EF0107', lw=2.5, markersize=6, label='UCL teams')
if 'Control' in monthly.columns:
    ax.plot(monthly.index.astype(str), monthly['Control'],
            's--', color='#0057B8', lw=2, markersize=5, label='Control')

# Mark UCL knockout months
for month in ['2025-02', '2025-03', '2025-04', '2025-05']:
    ax.axvspan(
        month, month,
        alpha=0.08, color='orange',
        label='UCL knockouts' if month == '2025-02' else ''
    )

ax.set(title='Monthly Average Decline Risk: UCL teams vs. Control (2024/25)',
       xlabel='Month', ylabel='Avg decline probability')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'risk_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Player risk table — trainer-facing output

In [ ]:
# Full test-set player risk summary
player_risk = (
    test_df
    .groupby(['team', 'player', 'position_group'])
    .agg(
        matches          = ('decline_flag', 'count'),
        actual_declines  = ('decline_flag', 'sum'),
        actual_decline_pct = ('decline_flag', 'mean'),
        avg_risk_score   = ('decline_probability', 'mean'),
        max_risk_score   = ('decline_probability', 'max'),
        avg_rest_days    = ('rest_days', 'mean'),
        short_rest_count = ('short_rest', 'sum'),
        ucl_matches      = ('is_ucl_match', 'sum'),
    )
    .round(3)
    .sort_values('avg_risk_score', ascending=False)
    .reset_index()
)

# Risk tier
def risk_tier(p):
    if p >= 0.50: return '🔴 HIGH'
    if p >= 0.35: return '🟡 MEDIUM'
    return '🟢 LOW'

player_risk['risk_tier'] = player_risk['avg_risk_score'].apply(risk_tier)

print('\n=== TRAINER-FACING PLAYER RISK TABLE (Top 25) ===')
print(player_risk.head(25).to_string(index=False))

# Save
player_risk.to_csv(OUTPUT_DIR / 'player_risk_table_xgb.csv', index=False)
print(f'\nSaved: {OUTPUT_DIR / "player_risk_table_xgb.csv"}')

# High-risk players
high_risk = player_risk[player_risk['avg_risk_score'] >= 0.35]
print(f'\nHigh-risk players (≥35%): {len(high_risk)}')
for _, r in high_risk.iterrows():
    print(f"  [{r['risk_tier']}] {r['player']:<25} {r['team']:<20} "
          f"avg_risk={r['avg_risk_score']:.2f}  rest={r['avg_rest_days']:.1f}d  "
          f"short_rest={r['short_rest_count']}x  ucl={r['ucl_matches']}")

## 13. Forward-looking: predict risk for next match

This is the operational output — given current rest and workload data, what is each player's risk for the next match?

In [ ]:
# Take each player's MOST RECENT observation as their current state
current_state = (
    df_model
    .sort_values('match_date')
    .groupby(['team','player'])
    .last()
    .reset_index()
)

# Only UCL teams for the coaching dashboard
current_ucl = current_state[current_state['is_ucl_team'] == 1].copy()

X_now = current_ucl[FEATURE_COLS].fillna(0)
current_ucl['next_match_risk'] = xgb_model.predict_proba(X_now)[:, 1]
current_ucl['risk_tier']       = current_ucl['next_match_risk'].apply(risk_tier)

# Sort by risk and show coaching dashboard
dashboard = current_ucl[
    ['team','player','position_group','next_match_risk','risk_tier',
     'rest_days','min_last_3','ucl_min_last_3','rolling_baseline']
].sort_values('next_match_risk', ascending=False).reset_index(drop=True)

print('\n' + '='*80)
print('  COACHING DASHBOARD — NEXT-MATCH RISK FORECAST')
print('  (Based on current rest and workload state)')
print('='*80)
print(dashboard.to_string(index=False))

# Save for coaching staff
dashboard.to_csv(OUTPUT_DIR / 'next_match_risk_dashboard.csv', index=False)
print(f'\nSaved: {OUTPUT_DIR / "next_match_risk_dashboard.csv"}')

## 14. Multi-season trend: has congestion effect grown over time?

In [ ]:
# Re-predict on full dataset (not just test)
X_all = df_model[FEATURE_COLS].fillna(0)
df_model['pred_risk'] = xgb_model.predict_proba(X_all)[:, 1]

season_group = df_model.groupby(['season','congestion_group']).agg(
    avg_risk      = ('pred_risk', 'mean'),
    decline_rate  = ('decline_flag', 'mean'),
    avg_rest      = ('rest_days', 'mean'),
    short_rest_pct = ('short_rest', 'mean'),
).round(3).reset_index()

print('\n=== MULTI-SEASON COMPARISON ===')
print(season_group.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ('avg_risk',       'Average predicted risk'),
    ('decline_rate',   'Actual decline rate'),
    ('short_rest_pct', 'Short-rest match %'),
]

for ax, (metric, title) in zip(axes, metrics):
    for group, color in [('UCL_team', '#EF0107'), ('Control', '#0057B8')]:
        subset = season_group[season_group['congestion_group'] == group]
        ax.plot(subset['season'], subset[metric],
                'o-', color=color, lw=2, markersize=7,
                label=group.replace('_', ' '))
    ax.set(title=title, xlabel='Season', ylabel=metric)
    ax.tick_params(axis='x', rotation=20)
    ax.legend(fontsize=8)

plt.suptitle('Multi-Season Trend: UCL teams vs. Control group', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'multi_season_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Save model and all outputs

In [ ]:
import joblib

# Save model
model_path = OUTPUT_DIR / 'xgb_decline_risk_model.json'
xgb_model.save_model(str(model_path))
print(f'Model saved: {model_path}')

# Save label encoders
joblib.dump(label_encoders, OUTPUT_DIR / 'label_encoders.pkl')

# Save feature list (important for inference later)
with open(OUTPUT_DIR / 'feature_cols.json', 'w') as f:
    import json
    json.dump(FEATURE_COLS, f)

# Save full test predictions
test_df.to_csv(OUTPUT_DIR / 'test_set_predictions.csv', index=False)

# Model metadata summary
print('\n' + '='*70)
print('MODEL SUMMARY')
print('='*70)
print(f'Model type          : XGBoost Classifier')
print(f'Seasons used        : {SEASONS}')
print(f'Train/test split    : 22/23+23/24 → train | 24/25 → test')
print(f'Features            : {len(FEATURE_COLS)}')
print(f'Training samples    : {len(X_train):,}')
print(f'Test samples        : {len(X_test):,}')
print(f'CV ROC-AUC          : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'Test ROC-AUC        : {test_auc:.3f}')
print(f'Test Avg Precision  : {test_ap:.3f}')
print(f'Output directory    : {OUTPUT_DIR}')
print('\nFiles saved:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file() and not f.name.startswith('.'):
        print(f'  {f.relative_to(OUTPUT_DIR)}')

## 16. Design notes and next steps

### Improvements over Notebook 03 (Logistic Regression)
- **XGBoost** captures non-linear interactions (e.g. short-rest × UCL load × player baseline)
- **3 seasons** of data triples sample size, making estimates more stable
- **Control group** (non-UCL PL teams) enables causal-style comparison as suggested by tutor
- **Grouped cross-validation** (teams as groups) prevents data leakage between folds
- **SHAP values** explain individual predictions — useful for coaching staff
- **Opponent ELO** included as predictor

### Limitations to acknowledge in the report
1. Congestion is not randomly assigned (best teams play more) — confounding is still present
   → The control group reduces but does not eliminate this
   → Consider propensity score matching in future work
2. Performance score is a proxy — no physical fatigue markers (GPS, HR)
3. Injury data is not included — this predicts performance *decline*, not injury
4. Aston Villa only in UCL for 24/25 — 2 seasons without UCL load will dilute their signal

### Recommended usage
1. Before each matchday, run cell 13 to refresh the risk dashboard
2. Flag players with `next_match_risk ≥ 0.35` for discussion with medical staff
3. Retrain on new season data after the season ends (cell 7)
4. Use SHAP waterfall plots for individual player explanations

### Potential additions for Bayesian hierarchical extension (tutor suggestion)
```python
# Future: replace XGBoost with PyMC hierarchical model
# import pymc as pm
# with pm.Model(coords={'team': teams, 'player': players}) as hier_model:
#     mu_team  = pm.Normal('mu_team', mu=0, sigma=1, dims='team')
#     sigma    = pm.HalfNormal('sigma', 0.5)
#     p_decline = pm.math.invlogit(mu_team[team_idx] + beta @ X)
#     obs = pm.Bernoulli('obs', p=p_decline, observed=y)
```